# Final Unified 3D Filter Visualization

This notebook loads one results folder, replays KF/EKF/UKF from the same detections, stores a unified per-timestep record table, and exposes one configurable Plotly figure builder for presentation figures.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "befr_visual_tracking" / "camera_model.py").is_file():
            return candidate
    raise RuntimeError("Could not find BEFR-Project repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from befr_visual_tracking.camera_model import Camera, cameras_from_yaml
from befr_visual_tracking.ekf_tracker import EkfTracker, predicted_pixel_measurement
from befr_visual_tracking.experiments.replay import load_detections_for_replay, load_ground_truth_for_replay
from befr_visual_tracking.kalman_tracker import KalmanTracker, linear_position_update
from befr_visual_tracking.result_writer import load_metadata_json
from befr_visual_tracking.tracker_base import TrackerDetection
from befr_visual_tracking.triangulation import PixelObservation, triangulate_from_observations
from befr_visual_tracking.ukf_core import generate_sigma_points, ukf_weights
from befr_visual_tracking.ukf_tracker import UkfTracker, project_state_to_pixel

REPO_ROOT


## Controls

Change `RESULTS_DIR` and `STATIC_TIMESTEP_INDEX` to inspect a different run or time step. The notebook runs all filters from the same detections.


In [ ]:
RESULTS_DIR = REPO_ROOT / "results" / "three_camera_takeoff_circles4"
STATIC_TIMESTEP_INDEX = 73

FRUSTUM_DEPTH_M = 1.4
GRID_STEP_PX = 1
PLOT_HEIGHT = 760
PLOT_ASPECTMODE = "data"
STATIC_HTML_OUTPUT_DIR = REPO_ROOT / "befr_visual_tracking" / "notebooks" / "final_vis_figures" / RESULTS_DIR.name
STATIC_HTML_INCLUDE_PLOTLYJS = True

CLOSE_UP_EYE = {"x": 0.25, "y": -0.25, "z": 0.9}
CLOSE_UP_CENTER = {"x": -0.15, "y": 0.15, "z": 0.25}
CLOSE_UP_UP = {"x": 0.0, "y": 0.0, "z": 1.0}
PLOT_SCENE_CAMERA = {"eye": CLOSE_UP_EYE, "center": CLOSE_UP_CENTER, "up": CLOSE_UP_UP}
FIX_SCENE_AXIS_RANGES = True
SCENE_AXIS_PADDING_M = 0.35
SCENE_AXIS_RANGES = None  # Computed from cameras, trajectory, and filter states after replay.

FILTER_ACCELERATION_NOISE_STD = 5.0
FILTER_INNOVATION_GATE = 9.21
FILTER_MONTE_CARLO_SAMPLES = 50
FILTER_RANDOM_SEED = None  # None uses metadata random_seed.
UKF_ALPHA = 1 / np.sqrt(6)
UKF_BETA = 2.0
UKF_KAPPA = 0.0

TRIANGULATION_MONTE_CARLO_SAMPLE_COUNT = 40
TRIANGULATION_MONTE_CARLO_SEED = 123
TRIANGULATION_DETECTION_RAY_LENGTH_M = 5.0

SHOW_PIXEL_GRID = True
MEASUREMENT_SIGMA_LEVELS = [3.0, 2.0, 1.0]
PROJECTION_SIGMA_LEVELS = [3.0, 2.0, 1.0]
POSITION_ELLIPSOID_SIGMA = 1.0
VELOCITY_ARROW_TIME_HORIZON_S = 0.5

TRUE_COLOR = "#20c05c"
DETECTION_COLOR = "red"
KF_COLOR = "#f1611b"
KF_COVARIANCE_COLOR = "#f59e0b"
TRIANGULATION_POINT_COLOR = "#b45309"
EKF_COLOR = "dodgerblue"
UKF_COLOR = "#7c3aed"
UKF_MEASUREMENT_COLOR = "#7c3aed"
PREVIOUS_ALPHA_COLOR = "#64748b"

CAMERA_COLORS = {
    "camera_0": "#f75ab0",
    "camera_1": "#ae3ae8",
    "camera_2": "#f68665",
    "camera_3": "#509e60",
}
DEFAULT_CAMERA_COLOR = "#00a6a6"
FILTER_COLORS = {"kf": KF_COLOR, "ekf": EKF_COLOR, "ukf": UKF_COLOR}
FILTER_LABELS = {"kf": "KF", "ekf": "EKF", "ukf": "UKF"}


## Geometry and Drawing Helpers


In [ ]:
def image_plane_point(camera: Camera, u: float, v: float, depth_m: float = FRUSTUM_DEPTH_M) -> np.ndarray:
    origin, direction_world = camera.pixel_to_world_ray(float(u), float(v))
    direction_camera = camera.extrinsics.R_world_to_camera @ direction_world
    z_component = float(direction_camera[2])
    if abs(z_component) < 1e-12:
        raise ValueError(f"Pixel ray for {camera.name} is parallel to the image plane")
    return origin + (float(depth_m) / z_component) * direction_world


def image_plane_corners(camera: Camera, depth_m: float = FRUSTUM_DEPTH_M) -> list[np.ndarray]:
    width = float(camera.intrinsics.width_px)
    height = float(camera.intrinsics.height_px)
    return [
        image_plane_point(camera, 0.0, 0.0, depth_m),
        image_plane_point(camera, width, 0.0, depth_m),
        image_plane_point(camera, width, height, depth_m),
        image_plane_point(camera, 0.0, height, depth_m),
    ]


def line_trace(points: list[np.ndarray | None], *, color: str, width: float, name: str, showlegend: bool = False, opacity: float = 1.0) -> go.Scatter3d:
    return go.Scatter3d(
        x=[float(p[0]) if p is not None else None for p in points],
        y=[float(p[1]) if p is not None else None for p in points],
        z=[float(p[2]) if p is not None else None for p in points],
        mode="lines",
        line={"color": color, "width": width},
        opacity=opacity,
        name=name,
        showlegend=showlegend,
        hoverinfo="name",
    )


def camera_index_label(camera: Camera) -> str:
    suffix = camera.name.rsplit("_", 1)[-1]
    return suffix if suffix.isdigit() else camera.name


def add_camera_frustum(fig: go.Figure, camera: Camera, color: str, depth_m: float = FRUSTUM_DEPTH_M) -> None:
    origin = camera.extrinsics.t_camera_to_world
    corners = image_plane_corners(camera, depth_m)
    fig.add_trace(go.Scatter3d(x=[origin[0]], y=[origin[1]], z=[origin[2]], mode="markers", marker={"size": 5, "color": color}, name=camera.name, hovertemplate=f"{camera.name}<br>x=%{{x:.2f}} m<br>y=%{{y:.2f}} m<br>z=%{{z:.2f}} m<extra></extra>"))
    fig.add_trace(go.Mesh3d(x=[p[0] for p in corners], y=[p[1] for p in corners], z=[p[2] for p in corners], i=[0, 0], j=[1, 2], k=[2, 3], color=color, opacity=0.10, name=f"{camera.name} image plane", showlegend=False, hoverinfo="skip"))
    fig.add_trace(line_trace([*corners, corners[0]], color=color, width=4, name=f"{camera.name} image border"))
    frustum_lines: list[np.ndarray | None] = []
    for corner in corners:
        frustum_lines.extend([origin, corner, None])
    fig.add_trace(line_trace(frustum_lines, color=color, width=2, name=f"{camera.name} frustum"))


def add_pixel_grid(fig: go.Figure, camera: Camera, depth_m: float, step_px: int, color: str) -> None:
    width = int(camera.intrinsics.width_px)
    height = int(camera.intrinsics.height_px)
    step_px = max(1, int(step_px))
    u_values = list(range(0, width + 1, step_px))
    v_values = list(range(0, height + 1, step_px))
    if u_values[-1] != width:
        u_values.append(width)
    if v_values[-1] != height:
        v_values.append(height)
    grid_lines: list[np.ndarray | None] = []
    for u in u_values:
        grid_lines.extend([image_plane_point(camera, u, 0.0, depth_m), image_plane_point(camera, u, height, depth_m), None])
    for v in v_values:
        grid_lines.extend([image_plane_point(camera, 0.0, v, depth_m), image_plane_point(camera, width, v, depth_m), None])
    fig.add_trace(line_trace(grid_lines, color=color, width=1.2, name=f"{camera.name} pixel grid"))


def add_image_axis_glyph(fig: go.Figure, camera: Camera, depth_m: float = FRUSTUM_DEPTH_M) -> None:
    width = float(camera.intrinsics.width_px)
    height = float(camera.intrinsics.height_px)
    small_dim = min(width, height)
    outside_px = max(1.2, 0.10 * small_dim)
    axis_length_px = max(2.4, 0.30 * small_dim)
    label_offset_px = max(1.8, 0.20 * small_dim)
    origin_uv = (-outside_px, height + outside_px)
    u_tip_uv = (origin_uv[0] + axis_length_px, origin_uv[1])
    v_tip_uv = (origin_uv[0], origin_uv[1] - axis_length_px)
    origin = image_plane_point(camera, origin_uv[0], origin_uv[1], depth_m)
    u_tip = image_plane_point(camera, u_tip_uv[0], u_tip_uv[1], depth_m)
    v_tip = image_plane_point(camera, v_tip_uv[0], v_tip_uv[1], depth_m)
    u_label = image_plane_point(camera, origin_uv[0] + 0.55 * axis_length_px, origin_uv[1] + 0.55 * label_offset_px, depth_m)
    v_label = image_plane_point(camera, origin_uv[0] - 0.85 * label_offset_px, origin_uv[1] - 0.70 * axis_length_px, depth_m)
    fig.add_trace(line_trace([origin, u_tip, None, origin, v_tip], color="black", width=5, name=f"{camera.name} image uv axes"))
    for tip, vector, axis_name in [(u_tip, u_tip - origin, "u"), (v_tip, v_tip - origin, "v")]:
        vector_norm = float(np.linalg.norm(vector))
        if vector_norm < 1e-12:
            continue
        fig.add_trace(go.Cone(x=[tip[0]], y=[tip[1]], z=[tip[2]], u=[vector[0]], v=[vector[1]], w=[vector[2]], sizemode="absolute", sizeref=0.18 * vector_norm, anchor="tip", colorscale=[[0.0, "black"], [1.0, "black"]], showscale=False, name=f"{camera.name} {axis_name} image-axis arrowhead", showlegend=False, hoverinfo="skip"))
    camera_index = camera_index_label(camera)
    fig.add_trace(go.Scatter3d(x=[u_label[0]], y=[u_label[1]], z=[u_label[2]], mode="text", text=[f"u_{camera_index}"], textfont={"color": "black", "size": 11}, name=f"{camera.name} image u label", showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter3d(x=[v_label[0]], y=[v_label[1]], z=[v_label[2]], mode="text", text=[f"v_{camera_index}"], textfont={"color": "black", "size": 11}, name=f"{camera.name} image v label", showlegend=False, hoverinfo="skip"))


def add_camera_geometry(fig: go.Figure, cameras: list[Camera]) -> None:
    for camera in cameras:
        color = CAMERA_COLORS.get(camera.name, DEFAULT_CAMERA_COLOR)
        add_camera_frustum(fig, camera, color)
        if SHOW_PIXEL_GRID:
            add_pixel_grid(fig, camera, FRUSTUM_DEPTH_M, GRID_STEP_PX, color)
        add_image_axis_glyph(fig, camera, FRUSTUM_DEPTH_M)


In [ ]:
def velocity_arrow_geometry(origin: np.ndarray, velocity: np.ndarray, time_horizon_s: float = VELOCITY_ARROW_TIME_HORIZON_S):
    origin = np.asarray(origin, dtype=float).reshape(3)
    velocity = np.asarray(velocity, dtype=float).reshape(3)
    arrow_vector = float(time_horizon_s) * velocity
    endpoint = origin + arrow_vector
    speed = float(np.linalg.norm(arrow_vector))
    if speed < 1e-12:
        return origin, origin, endpoint, 0.01, arrow_vector
    direction = arrow_vector / speed
    cone_length = min(0.18, 0.35 * speed)
    shaft_endpoint = endpoint - cone_length * direction
    return origin, shaft_endpoint, endpoint, cone_length, arrow_vector


def add_velocity_arrow(fig: go.Figure, origin: np.ndarray, velocity: np.ndarray, *, color: str, name: str, width: float = 6) -> None:
    origin, shaft_endpoint, endpoint, cone_length, arrow_vector = velocity_arrow_geometry(origin, velocity)
    fig.add_trace(go.Scatter3d(x=[origin[0], shaft_endpoint[0]], y=[origin[1], shaft_endpoint[1]], z=[origin[2], shaft_endpoint[2]], mode="lines", line={"color": color, "width": width}, name=name, showlegend=False))
    fig.add_trace(go.Cone(x=[endpoint[0]], y=[endpoint[1]], z=[endpoint[2]], u=[arrow_vector[0]], v=[arrow_vector[1]], w=[arrow_vector[2]], sizemode="absolute", sizeref=cone_length, anchor="tip", colorscale=[[0.0, color], [1.0, color]], showscale=False, name=f"{name} arrowhead", showlegend=False))


def add_state_marker(fig: go.Figure, position: np.ndarray, velocity: np.ndarray | None, *, color: str, name: str, symbol: str = "circle", size: int = 8, show_velocity: bool = False) -> None:
    position = np.asarray(position, dtype=float).reshape(3)
    fig.add_trace(go.Scatter3d(x=[position[0]], y=[position[1]], z=[position[2]], mode="markers", marker={"size": size, "color": color, "symbol": symbol}, name=name, hovertemplate=f"{name}<br>x=%{{x:.3f}} m<br>y=%{{y:.3f}} m<br>z=%{{z:.3f}} m<extra></extra>"))
    if show_velocity and velocity is not None:
        add_velocity_arrow(fig, position, np.asarray(velocity, dtype=float).reshape(3), color=color, name=f"{name} velocity")


def add_projection_ray(fig: go.Figure, camera: Camera, point_world: np.ndarray, *, color: str, name: str, width: float = 4) -> tuple[float, float] | None:
    uv = camera.project_world_point_for_filter(np.asarray(point_world, dtype=float).reshape(3))
    if uv is None:
        return None
    origin, direction = camera.pixel_to_world_ray(*uv)
    point_world = np.asarray(point_world, dtype=float).reshape(3)
    distance = float(np.linalg.norm(point_world - origin))
    endpoint = origin + distance * direction
    if np.dot(endpoint - origin, point_world - origin) > 0.0:
        endpoint = point_world
    fig.add_trace(go.Scatter3d(x=[origin[0], endpoint[0]], y=[origin[1], endpoint[1]], z=[origin[2], endpoint[2]], mode="lines", line={"color": color, "width": width}, name=name, showlegend=False, hovertemplate=f"{name}<br>u={uv[0]:.2f}px<br>v={uv[1]:.2f}px<extra></extra>"))
    return float(uv[0]), float(uv[1])


def add_image_marker(fig: go.Figure, camera: Camera, uv: tuple[float, float] | np.ndarray, *, color: str, name: str, size: int = 5, symbol: str = "circle") -> None:
    point = image_plane_point(camera, float(uv[0]), float(uv[1]), FRUSTUM_DEPTH_M)
    fig.add_trace(go.Scatter3d(x=[point[0]], y=[point[1]], z=[point[2]], mode="markers", marker={"size": size, "color": color, "symbol": symbol}, name=name, showlegend=False, hovertemplate=f"{name}<br>u={float(uv[0]):.2f}px<br>v={float(uv[1]):.2f}px<extra></extra>"))


def quantized_pixel_corners(camera: Camera, uv: tuple[float, float] | np.ndarray, depth_m: float = FRUSTUM_DEPTH_M) -> list[np.ndarray]:
    u = float(np.floor(float(uv[0])))
    v = float(np.floor(float(uv[1])))
    return [
        image_plane_point(camera, u, v, depth_m),
        image_plane_point(camera, u + 1.0, v, depth_m),
        image_plane_point(camera, u + 1.0, v + 1.0, depth_m),
        image_plane_point(camera, u, v + 1.0, depth_m),
    ]


def add_quantized_pixel_cell(fig: go.Figure, camera: Camera, uv: tuple[float, float] | np.ndarray, *, color: str = DETECTION_COLOR) -> None:
    corners = quantized_pixel_corners(camera, uv, FRUSTUM_DEPTH_M)
    surface_rows = [[corners[0], corners[1]], [corners[3], corners[2]]]
    fig.add_trace(go.Surface(x=[[p[0] for p in row] for row in surface_rows], y=[[p[1] for p in row] for row in surface_rows], z=[[p[2] for p in row] for row in surface_rows], surfacecolor=np.ones((2, 2)), colorscale=[[0.0, color], [1.0, color]], opacity=0.70, showscale=False, name=f"{camera.name} quantized detection pixel", showlegend=False))
    fig.add_trace(line_trace([corners[0], corners[1], None, corners[1], corners[2], None, corners[2], corners[3], None, corners[3], corners[0]], color=color, width=3, name=f"{camera.name} quantized detection outline"))


def covariance_ellipsoid_mesh(center: np.ndarray, covariance_3x3: np.ndarray, sigma_scale: float, *, num_u: int = 28, num_v: int = 14):
    covariance_3x3 = np.asarray(covariance_3x3, dtype=float).reshape(3, 3)
    covariance_3x3 = 0.5 * (covariance_3x3 + covariance_3x3.T)
    eigvals, eigvecs = np.linalg.eigh(covariance_3x3)
    eigvals = np.clip(eigvals, 1e-12, None)
    radii = float(sigma_scale) * np.sqrt(eigvals)
    u = np.linspace(0.0, 2.0 * np.pi, num_u)
    v = np.linspace(0.0, np.pi, num_v)
    unit_x = np.outer(np.cos(u), np.sin(v))
    unit_y = np.outer(np.sin(u), np.sin(v))
    unit_z = np.outer(np.ones_like(u), np.cos(v))
    unit_points = np.stack([unit_x, unit_y, unit_z], axis=-1)
    scaled = unit_points @ np.diag(radii) @ eigvecs.T
    surface = scaled + np.asarray(center, dtype=float).reshape(3)
    return surface[..., 0], surface[..., 1], surface[..., 2], eigvals


def add_position_covariance_ellipsoid(fig: go.Figure, position: np.ndarray, covariance: np.ndarray, *, color: str, name: str, sigma: float = POSITION_ELLIPSOID_SIGMA, opacity: float = 0.22, num_u: int = 28, num_v: int = 14) -> None:
    x, y, z, _ = covariance_ellipsoid_mesh(position, np.asarray(covariance, dtype=float)[:3, :3], sigma, num_u=num_u, num_v=num_v)
    fig.add_trace(go.Surface(x=x, y=y, z=z, surfacecolor=np.ones_like(x), colorscale=[[0.0, color], [1.0, color]], opacity=opacity, showscale=False, name=name, showlegend=False))


In [ ]:
def pixel_covariance_ellipse_points(camera: Camera, center_uv: tuple[float, float] | np.ndarray, covariance_2x2: np.ndarray, sigma_scale: float, depth_m: float = FRUSTUM_DEPTH_M, num_points: int = 72) -> np.ndarray:
    covariance_2x2 = np.asarray(covariance_2x2, dtype=float).reshape(2, 2)
    covariance_2x2 = 0.5 * (covariance_2x2 + covariance_2x2.T)
    eigvals, eigvecs = np.linalg.eigh(covariance_2x2)
    eigvals = np.clip(eigvals, 1e-12, None)
    theta = np.linspace(0.0, 2.0 * np.pi, num_points, endpoint=False)
    unit_circle = np.vstack([np.cos(theta), np.sin(theta)])
    offsets = eigvecs @ (np.diag(float(sigma_scale) * np.sqrt(eigvals)) @ unit_circle)
    uv_points = np.asarray(center_uv, dtype=float).reshape(2, 1) + offsets
    return np.array([image_plane_point(camera, u, v, depth_m) for u, v in uv_points.T], dtype=float)


def add_pixel_covariance_ellipses(fig: go.Figure, camera: Camera, center_uv: tuple[float, float] | np.ndarray, covariance_2x2: np.ndarray, levels: list[float], *, color: str, name_prefix: str) -> None:
    center = image_plane_point(camera, float(center_uv[0]), float(center_uv[1]), FRUSTUM_DEPTH_M)
    valid_levels = sorted([float(level) for level in levels if level > 0.0])
    max_level = max(valid_levels) if valid_levels else 1.0
    for sigma_scale in valid_levels[::-1]:
        ring = pixel_covariance_ellipse_points(camera, center_uv, covariance_2x2, sigma_scale, FRUSTUM_DEPTH_M)
        points = np.vstack([center, ring])
        num_ring = len(ring)
        opacity = 0.07 + 0.23 * (1.0 - (sigma_scale - 1.0) / max(max_level - 1.0, 1.0))
        fig.add_trace(go.Mesh3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], i=np.zeros(num_ring, dtype=int), j=np.arange(1, num_ring + 1, dtype=int), k=np.array([idx + 1 if idx < num_ring else 1 for idx in range(1, num_ring + 1)], dtype=int), color=color, opacity=opacity, name=f"{camera.name} {name_prefix} {sigma_scale:.0f} sigma", showlegend=False))


def state_position_pixel_covariance(camera: Camera, position: np.ndarray, covariance_6x6: np.ndarray) -> np.ndarray | None:
    projected = camera.project_world_point_for_filter(np.asarray(position, dtype=float).reshape(3))
    if projected is None:
        return None
    J = camera.projection_jacobian(position)
    P = np.asarray(covariance_6x6, dtype=float)[:3, :3]
    S = J @ P @ J.T
    return 0.5 * (S + S.T)


def apply_layout(fig: go.Figure, title: str, timestamp: float | None = None) -> None:
    title_text = title if timestamp is None else f"{title} - t = {float(timestamp):.2f} s"
    scene = {
        "xaxis_title": "X [m]",
        "yaxis_title": "Y [m]",
        "zaxis_title": "Z [m]",
        "aspectmode": PLOT_ASPECTMODE,
        "camera": PLOT_SCENE_CAMERA,
    }
    if FIX_SCENE_AXIS_RANGES and SCENE_AXIS_RANGES is not None:
        scene.update({
            "xaxis": {"title": "X [m]", "range": SCENE_AXIS_RANGES["x"], "autorange": False},
            "yaxis": {"title": "Y [m]", "range": SCENE_AXIS_RANGES["y"], "autorange": False},
            "zaxis": {"title": "Z [m]", "range": SCENE_AXIS_RANGES["z"], "autorange": False},
        })
    fig.update_layout(
        title=title_text,
        scene=scene,
        margin={"l": 0, "r": 0, "t": 50, "b": 0},
        height=PLOT_HEIGHT,
        legend={"orientation": "h", "yanchor": "bottom", "y": 0.01, "xanchor": "left", "x": 0.01},
    )


## Load Data and Replay Filters


In [ ]:
metadata = load_metadata_json(RESULTS_DIR / "metadata.json")
_, cameras = cameras_from_yaml(RESULTS_DIR / "camera_config.yaml")
cameras_by_id = {camera.name: camera for camera in cameras}
ACTIVE_CAMERA_IDS = list(metadata.get("active_cameras", [camera.name for camera in cameras]))
selected_cameras = [cameras_by_id[camera_id] for camera_id in ACTIVE_CAMERA_IDS]
FILTER_RANDOM_SEED = int(metadata.get("random_seed", 42)) if FILTER_RANDOM_SEED is None else int(FILTER_RANDOM_SEED)

detections_by_time = load_detections_for_replay(RESULTS_DIR)
ground_truth_by_time = load_ground_truth_for_replay(RESULTS_DIR)
ideal_detections = pd.read_csv(RESULTS_DIR / "detections_ideal.csv") if (RESULTS_DIR / "detections_ideal.csv").exists() else pd.DataFrame()
ideal_uv_by_time_camera = {
    (float(row.timestamp), row.camera_id): (float(row.u_ideal), float(row.v_ideal))
    for row in ideal_detections.itertuples(index=False)
    if bool(row.visible)
}


def detection_uv_for_camera(record: dict, camera: Camera) -> tuple[float, float] | None:
    for detection in record["detections"]:
        if detection.camera_id == camera.name:
            return float(detection.u), float(detection.v)
    return None


def ideal_uv_for_camera(record: dict, camera: Camera) -> tuple[float, float] | None:
    timestamp = float(record["timestamp"])
    ideal = ideal_uv_by_time_camera.get((timestamp, camera.name))
    if ideal is not None:
        return ideal
    projected = camera.project_world_point_for_filter(record["ground_truth_position"])
    if projected is None:
        return None
    return float(projected[0]), float(projected[1])


def scalar_pixel_noise_std() -> float:
    if not selected_cameras:
        return 1.0
    camera = selected_cameras[0]
    return 0.5 * (float(camera.noise_model.pixel_noise_std_u) + float(camera.noise_model.pixel_noise_std_v))


def observations_for_detections(detections: list[TrackerDetection]) -> list[PixelObservation]:
    active = set(ACTIVE_CAMERA_IDS)
    return [
        PixelObservation(camera_id=detection.camera_id, u=float(detection.u), v=float(detection.v))
        for detection in detections
        if detection.camera_id in active
    ]


def triangulation_result_for_detections(detections: list[TrackerDetection], rng: np.random.Generator, *, monte_carlo_samples: int = FILTER_MONTE_CARLO_SAMPLES):
    observations = observations_for_detections(detections)
    if len(observations) < 2:
        return None
    return triangulate_from_observations(
        cameras_by_id,
        observations,
        estimate_covariance=True,
        pixel_noise_std=scalar_pixel_noise_std(),
        monte_carlo_samples=monte_carlo_samples,
        rng=rng,
    )


def triangulation_acceptable(result) -> bool:
    if result is None or result.point is None or result.covariance is None:
        return False
    if not result.quality.is_valid:
        return False
    if result.quality.reprojection_rmse > 10.0:
        return False
    if result.quality.ray_intersection_residual > 1.0:
        return False
    return True


def triangulation_monte_carlo_samples_for_detections(detections: list[TrackerDetection], sample_count: int = TRIANGULATION_MONTE_CARLO_SAMPLE_COUNT, seed: int = TRIANGULATION_MONTE_CARLO_SEED) -> np.ndarray:
    observations = observations_for_detections(detections)
    if len(observations) < 2 or sample_count <= 0:
        return np.empty((0, 3), dtype=float)
    rng = np.random.default_rng(int(seed))
    sigma = scalar_pixel_noise_std()
    points: list[np.ndarray] = []
    attempts = 0
    max_attempts = max(int(sample_count) * 5, int(sample_count) + 10)
    while len(points) < int(sample_count) and attempts < max_attempts:
        attempts += 1
        noisy = [
            PixelObservation(
                camera_id=observation.camera_id,
                u=observation.u + float(rng.normal(0.0, sigma)),
                v=observation.v + float(rng.normal(0.0, sigma)),
            )
            for observation in observations
        ]
        result = triangulate_from_observations(cameras_by_id, noisy, estimate_covariance=False)
        if result is not None and result.point is not None:
            points.append(result.point)
    if not points:
        return np.empty((0, 3), dtype=float)
    return np.asarray(points, dtype=float)


In [ ]:
def measurement_covariance_for_camera(camera: Camera, pixel_noise_std: float | None = None) -> np.ndarray:
    if pixel_noise_std is not None:
        sigma = float(pixel_noise_std)
        return np.diag([sigma**2, sigma**2])
    return camera.noise_model.measurement_noise_covariance


def empty_filter_record() -> dict[str, Any]:
    return {
        "valid": False,
        "previous_posterior_state": None,
        "previous_posterior_covariance": None,
        "prior_state": None,
        "prior_covariance": None,
        "posterior_state": None,
        "posterior_covariance": None,
        "ukf_bundle": None,
        "ukf_projected": {},
        "ekf_projected": {},
    }


def make_filter_record(tracker, *, previous_state, previous_covariance, prior_state, prior_covariance, extra: dict | None = None) -> dict[str, Any]:
    record = {
        "valid": True,
        "previous_posterior_state": None if previous_state is None else previous_state.copy(),
        "previous_posterior_covariance": None if previous_covariance is None else previous_covariance.copy(),
        "prior_state": prior_state.copy(),
        "prior_covariance": prior_covariance.copy(),
        "posterior_state": tracker.get_state(),
        "posterior_covariance": tracker.get_covariance(),
        "ukf_bundle": None,
        "ukf_projected": {},
        "ekf_projected": {},
    }
    if extra:
        record.update(extra)
    return record


def ukf_sigma_point_bundle_from_prior(prior_state: np.ndarray, prior_covariance: np.ndarray) -> dict[str, Any]:
    sigma_points, lambda_, alpha, beta = generate_sigma_points(prior_state, prior_covariance, alpha=UKF_ALPHA, beta=UKF_BETA, kappa=UKF_KAPPA)
    mean_weights, cov_weights = ukf_weights(lambda_, alpha=alpha, beta=beta)
    return {"state": prior_state.copy(), "covariance": prior_covariance.copy(), "sigma_points": sigma_points, "mean_weights": mean_weights, "cov_weights": cov_weights}


def projected_sigma_points_for_camera(camera: Camera, bundle: dict[str, Any]) -> dict[str, Any]:
    sigma_points = bundle["sigma_points"]
    projected_points = [project_state_to_pixel(camera, sigma_state) for sigma_state in sigma_points]
    valid_indices = [index for index, point in enumerate(projected_points) if point is not None]
    if not valid_indices or 0 not in valid_indices:
        return {"projected_points": projected_points, "valid_indices": valid_indices, "measurement_mean": None, "measurement_covariance": None}

    mean_weights = np.asarray(bundle["mean_weights"])[valid_indices].astype(float)
    cov_weights = np.asarray(bundle["cov_weights"])[valid_indices].astype(float)
    mean_weights /= np.sum(mean_weights)
    cov_weights /= np.sum(cov_weights)

    measurement_mean = np.zeros(2, dtype=float)
    for weight, index in zip(mean_weights, valid_indices, strict=True):
        measurement_mean += weight * projected_points[index]

    measurement_covariance = np.zeros((2, 2), dtype=float)
    for weight, index in zip(cov_weights, valid_indices, strict=True):
        delta = projected_points[index] - measurement_mean
        measurement_covariance += weight * np.outer(delta, delta)

    return {"projected_points": projected_points, "valid_indices": valid_indices, "measurement_mean": measurement_mean, "measurement_covariance": measurement_covariance}


def ekf_projected_prior_for_camera(camera: Camera, prior_state: np.ndarray, prior_covariance: np.ndarray) -> dict[str, Any]:
    prediction = predicted_pixel_measurement(camera, prior_state)
    if prediction is None:
        return {"measurement_mean": None, "measurement_covariance": None, "jacobian": None}
    measurement_mean, jacobian = prediction
    measurement_covariance = jacobian @ prior_covariance @ jacobian.T
    measurement_covariance = 0.5 * (measurement_covariance + measurement_covariance.T)
    return {"measurement_mean": measurement_mean, "measurement_covariance": measurement_covariance, "jacobian": jacobian}


def update_kf_tracker(tracker: KalmanTracker, timestamp: float, detections: list[TrackerDetection], triangulation_result) -> dict[str, Any]:
    previous_state = tracker.get_state() if tracker.is_initialized else None
    previous_covariance = tracker.get_covariance() if tracker.is_initialized else None

    if not tracker.is_initialized:
        if not triangulation_acceptable(triangulation_result):
            return empty_filter_record()
        tracker.initialize(triangulation_result.point, triangulation_result.covariance, timestamp)
        tracker.stats.num_initializations += 1
        prior_state = tracker.get_state()
        prior_covariance = tracker.get_covariance()
        return make_filter_record(tracker, previous_state=previous_state, previous_covariance=previous_covariance, prior_state=prior_state, prior_covariance=prior_covariance)

    tracker.predict(timestamp)
    prior_state = tracker.get_state()
    prior_covariance = tracker.get_covariance()
    if triangulation_acceptable(triangulation_result):
        posterior_state, posterior_covariance = linear_position_update(prior_state, prior_covariance, triangulation_result.point, triangulation_result.covariance)
        tracker._state = posterior_state
        tracker._covariance = posterior_covariance
        tracker.stats.num_updates += 1
    else:
        tracker.stats.num_prediction_only_steps += 1
    return make_filter_record(tracker, previous_state=previous_state, previous_covariance=previous_covariance, prior_state=prior_state, prior_covariance=prior_covariance)


def update_ekf_tracker(tracker: EkfTracker, timestamp: float, detections: list[TrackerDetection]) -> dict[str, Any]:
    previous_state = tracker.get_state() if tracker.is_initialized else None
    previous_covariance = tracker.get_covariance() if tracker.is_initialized else None
    if not tracker.is_initialized:
        tracker.try_initialize(timestamp, detections)
        if not tracker.is_initialized:
            return empty_filter_record()
        prior_state = tracker.get_state()
        prior_covariance = tracker.get_covariance()
    else:
        tracker.predict(timestamp)
        prior_state = tracker.get_state()
        prior_covariance = tracker.get_covariance()
    ekf_projected = {camera.name: ekf_projected_prior_for_camera(camera, prior_state, prior_covariance) for camera in selected_cameras}
    tracker.update(detections)
    return make_filter_record(tracker, previous_state=previous_state, previous_covariance=previous_covariance, prior_state=prior_state, prior_covariance=prior_covariance, extra={"ekf_projected": ekf_projected})


def update_ukf_tracker(tracker: UkfTracker, timestamp: float, detections: list[TrackerDetection]) -> dict[str, Any]:
    previous_state = tracker.get_state() if tracker.is_initialized else None
    previous_covariance = tracker.get_covariance() if tracker.is_initialized else None
    if not tracker.is_initialized:
        tracker.try_initialize(timestamp, detections)
        if not tracker.is_initialized:
            return empty_filter_record()
        prior_state = tracker.get_state()
        prior_covariance = tracker.get_covariance()
    else:
        tracker.predict(timestamp)
        prior_state = tracker.get_state()
        prior_covariance = tracker.get_covariance()
    bundle = ukf_sigma_point_bundle_from_prior(prior_state, prior_covariance)
    ukf_projected = {camera.name: projected_sigma_points_for_camera(camera, bundle) for camera in selected_cameras}
    tracker.update(detections)
    return make_filter_record(tracker, previous_state=previous_state, previous_covariance=previous_covariance, prior_state=prior_state, prior_covariance=prior_covariance, extra={"ukf_bundle": bundle, "ukf_projected": ukf_projected})


def run_all_filter_replays() -> list[dict[str, Any]]:
    kf_tracker = KalmanTracker(cameras_by_id, acceleration_noise_std=FILTER_ACCELERATION_NOISE_STD, active_cameras=ACTIVE_CAMERA_IDS, pixel_noise_std=scalar_pixel_noise_std(), monte_carlo_samples=FILTER_MONTE_CARLO_SAMPLES, rng=np.random.default_rng(FILTER_RANDOM_SEED))
    ekf_tracker = EkfTracker(cameras_by_id, acceleration_noise_std=FILTER_ACCELERATION_NOISE_STD, active_cameras=ACTIVE_CAMERA_IDS, pixel_noise_std=None, innovation_gate=FILTER_INNOVATION_GATE, monte_carlo_samples=FILTER_MONTE_CARLO_SAMPLES, rng=np.random.default_rng(FILTER_RANDOM_SEED))
    ukf_tracker = UkfTracker(cameras_by_id, acceleration_noise_std=FILTER_ACCELERATION_NOISE_STD, active_cameras=ACTIVE_CAMERA_IDS, pixel_noise_std=None, innovation_gate=FILTER_INNOVATION_GATE, monte_carlo_samples=FILTER_MONTE_CARLO_SAMPLES, alpha=UKF_ALPHA, beta=UKF_BETA, kappa=UKF_KAPPA, rng=np.random.default_rng(FILTER_RANDOM_SEED))
    triangulation_rng = np.random.default_rng(FILTER_RANDOM_SEED)

    records: list[dict[str, Any]] = []
    for timestamp in sorted(detections_by_time.keys()):
        gt = ground_truth_by_time.get(timestamp)
        if gt is None:
            continue
        detections = detections_by_time[timestamp]
        triangulation_result = triangulation_result_for_detections(detections, triangulation_rng)
        triangulation_samples = triangulation_monte_carlo_samples_for_detections(detections)
        record = {
            "index": len(records),
            "timestamp": timestamp,
            "detections": detections,
            "ground_truth_position": gt[0],
            "ground_truth_velocity": gt[1],
            "triangulation_result": triangulation_result,
            "triangulation_samples": triangulation_samples,
            "filters": {
                "kf": update_kf_tracker(kf_tracker, timestamp, detections, triangulation_result),
                "ekf": update_ekf_tracker(ekf_tracker, timestamp, detections),
                "ukf": update_ukf_tracker(ukf_tracker, timestamp, detections),
            },
        }
        if any(filter_record["valid"] for filter_record in record["filters"].values()):
            records.append(record)
    return records


unified_records = run_all_filter_replays()
assert unified_records, f"No initialized filter records produced for {RESULTS_DIR}"

def scene_axis_ranges_from_records(records: list[dict[str, Any]], padding_m: float = SCENE_AXIS_PADDING_M) -> dict[str, list[float]]:
    points: list[np.ndarray] = []
    for camera in selected_cameras:
        points.append(camera.extrinsics.t_camera_to_world)
        points.extend(image_plane_corners(camera, FRUSTUM_DEPTH_M))
    for record in records:
        points.append(record["ground_truth_position"])
        tri = record.get("triangulation_result")
        if tri is not None and tri.point is not None:
            points.append(tri.point)
        for filter_record in record["filters"].values():
            for key in ["previous_posterior_state", "prior_state", "posterior_state"]:
                state = filter_record.get(key)
                if state is not None:
                    points.append(np.asarray(state, dtype=float)[:3])
    arr = np.asarray(points, dtype=float)
    low = np.min(arr, axis=0) - float(padding_m)
    high = np.max(arr, axis=0) + float(padding_m)
    return {"x": [float(low[0]), float(high[0])], "y": [float(low[1]), float(high[1])], "z": [float(low[2]), float(high[2])]}


if SCENE_AXIS_RANGES is None:
    SCENE_AXIS_RANGES = scene_axis_ranges_from_records(unified_records)



In [ ]:
def flatten_state(prefix: str, state: np.ndarray | None, row: dict[str, Any]) -> None:
    names = ["x", "y", "z", "vx", "vy", "vz"]
    if state is None:
        for name in names:
            row[f"{prefix}_{name}"] = np.nan
        return
    state = np.asarray(state, dtype=float).reshape(6)
    for name, value in zip(names, state, strict=True):
        row[f"{prefix}_{name}"] = float(value)


def build_filter_records_dataframe(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for record in records:
        gt_pos = record["ground_truth_position"]
        gt_vel = record["ground_truth_velocity"]
        for filter_name, filter_record in record["filters"].items():
            row: dict[str, Any] = {
                "index": record["index"],
                "timestamp": record["timestamp"],
                "filter": filter_name,
                "valid": bool(filter_record["valid"]),
                "num_detections": len(record["detections"]),
                "gt_x": float(gt_pos[0]), "gt_y": float(gt_pos[1]), "gt_z": float(gt_pos[2]),
                "gt_vx": float(gt_vel[0]), "gt_vy": float(gt_vel[1]), "gt_vz": float(gt_vel[2]),
            }
            flatten_state("previous_posterior", filter_record.get("previous_posterior_state"), row)
            flatten_state("prior", filter_record.get("prior_state"), row)
            flatten_state("posterior", filter_record.get("posterior_state"), row)
            if filter_record.get("prior_state") is not None:
                row["prior_pos_error"] = float(np.linalg.norm(filter_record["prior_state"][:3] - gt_pos))
            else:
                row["prior_pos_error"] = np.nan
            if filter_record.get("posterior_state") is not None:
                row["posterior_pos_error"] = float(np.linalg.norm(filter_record["posterior_state"][:3] - gt_pos))
            else:
                row["posterior_pos_error"] = np.nan
            tri = record["triangulation_result"]
            row["triangulation_valid"] = bool(tri is not None and tri.point is not None and tri.covariance is not None)
            rows.append(row)
    return pd.DataFrame(rows)


filter_records_df = build_filter_records_dataframe(unified_records)
STATIC_TIMESTEP_INDEX = min(max(int(STATIC_TIMESTEP_INDEX), 0), len(unified_records) - 1)


def record_for_timestep(timestep_index: int | None = None) -> dict[str, Any]:
    index = STATIC_TIMESTEP_INDEX if timestep_index is None else int(timestep_index)
    index = min(max(index, 0), len(unified_records) - 1)
    return unified_records[index]


display(pd.DataFrame({"value": {
    "results_dir": str(RESULTS_DIR.relative_to(REPO_ROOT)),
    "trajectory": metadata.get("trajectory"),
    "active_cameras": ", ".join(ACTIVE_CAMERA_IDS),
    "num_unified_timesteps": len(unified_records),
    "static_timestep_index": STATIC_TIMESTEP_INDEX,
    "static_timestamp_s": record_for_timestep()["timestamp"],
    "camera_config": str((RESULTS_DIR / "camera_config.yaml").relative_to(REPO_ROOT)),
}}))
display(filter_records_df.head())


## Unified Static Figure Builder


In [ ]:
def add_detection_measurements(fig: go.Figure, record: dict[str, Any], *, show_points: bool, show_pixel_cells: bool, show_covariance: bool) -> None:
    for camera in selected_cameras:
        uv = detection_uv_for_camera(record, camera)
        if uv is None:
            continue
        if show_pixel_cells:
            add_quantized_pixel_cell(fig, camera, uv, color=DETECTION_COLOR)
        if show_points:
            add_image_marker(fig, camera, uv, color="limegreen", name=f"{camera.name} discrete measurement", size=4)
        if show_covariance:
            add_pixel_covariance_ellipses(fig, camera, uv, camera.noise_model.measurement_noise_covariance, MEASUREMENT_SIGMA_LEVELS, color=DETECTION_COLOR, name_prefix="measurement covariance around discrete measurement")


def add_ground_truth_overlays(fig: go.Figure, record: dict[str, Any], *, show_position: bool, show_velocity: bool, show_rays: bool, show_projection_points: bool, show_projection_covariance: bool) -> None:
    gt_pos = record["ground_truth_position"]
    gt_vel = record["ground_truth_velocity"]
    if show_position:
        add_state_marker(fig, gt_pos, gt_vel, color=TRUE_COLOR, name="ground truth", symbol="diamond", size=9, show_velocity=show_velocity)
    elif show_velocity:
        add_velocity_arrow(fig, gt_pos, gt_vel, color=TRUE_COLOR, name="ground truth velocity")
    for camera in selected_cameras:
        if show_rays:
            add_projection_ray(fig, camera, gt_pos, color=TRUE_COLOR, name=f"{camera.name} ray to ground truth", width=4)
        uv = ideal_uv_for_camera(record, camera)
        if uv is None:
            continue
        if show_projection_points:
            add_image_marker(fig, camera, uv, color=TRUE_COLOR, name=f"{camera.name} ground truth projection", size=4)
        if show_projection_covariance:
            add_pixel_covariance_ellipses(fig, camera, uv, camera.noise_model.measurement_noise_covariance, MEASUREMENT_SIGMA_LEVELS, color=TRUE_COLOR, name_prefix="ground truth projection measurement covariance")


def add_filter_state_3d(fig: go.Figure, filter_name: str, filter_record: dict[str, Any], state_key: str, covariance_key: str, *, position: bool, velocity: bool, covariance: bool, label: str, color: str) -> None:
    state = filter_record.get(state_key)
    cov = filter_record.get(covariance_key)
    if state is None:
        return
    if position:
        add_state_marker(fig, state[:3], state[3:], color=color, name=f"{FILTER_LABELS[filter_name]} {label}", symbol="circle", size=8, show_velocity=velocity)
    elif velocity:
        add_velocity_arrow(fig, state[:3], state[3:], color=color, name=f"{FILTER_LABELS[filter_name]} {label} velocity")
    if covariance and cov is not None:
        add_position_covariance_ellipsoid(fig, state[:3], cov, color=color, name=f"{FILTER_LABELS[filter_name]} {label} covariance")


def add_filter_projection_overlays(fig: go.Figure, filter_name: str, filter_record: dict[str, Any], state_key: str, covariance_key: str, *, show_rays: bool, show_points: bool, show_covariance: bool, color: str, label: str) -> None:
    state = filter_record.get(state_key)
    cov = filter_record.get(covariance_key)
    if state is None:
        return
    for camera in selected_cameras:
        if show_rays:
            add_projection_ray(fig, camera, state[:3], color=color, name=f"{camera.name} ray to {FILTER_LABELS[filter_name]} {label}", width=4)
        uv = camera.project_world_point_for_filter(state[:3])
        if uv is None:
            continue
        if show_points:
            add_image_marker(fig, camera, uv, color=color, name=f"{camera.name} {FILTER_LABELS[filter_name]} {label} projection", size=4)
        if show_covariance and cov is not None:
            pixel_cov = state_position_pixel_covariance(camera, state[:3], cov)
            if pixel_cov is not None:
                add_pixel_covariance_ellipses(fig, camera, uv, pixel_cov, PROJECTION_SIGMA_LEVELS, color=color, name_prefix=f"{FILTER_LABELS[filter_name]} {label} projected covariance")


def add_previous_to_prior_line(fig: go.Figure, filter_name: str, filter_record: dict[str, Any], color: str) -> None:
    previous = filter_record.get("previous_posterior_state")
    prior = filter_record.get("prior_state")
    if previous is None or prior is None:
        return
    fig.add_trace(line_trace([previous[:3], prior[:3]], color=color, width=5, name=f"{FILTER_LABELS[filter_name]} previous posterior to current prior"))


def add_ukf_sigma_overlays(fig: go.Figure, filter_record: dict[str, Any], *, show_3d: bool, show_pixel: bool, show_covariance_pixel: bool) -> None:
    bundle = filter_record.get("ukf_bundle")
    if bundle is None:
        return
    sigma_points = bundle["sigma_points"]
    if show_3d:
        fig.add_trace(go.Scatter3d(x=sigma_points[:, 0], y=sigma_points[:, 1], z=sigma_points[:, 2], mode="markers", marker={"size": 5, "color": UKF_COLOR, "symbol": "circle-open"}, name="UKF sigma points", hovertemplate="UKF sigma point<br>x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}<extra></extra>"))
    for camera in selected_cameras:
        projected = filter_record.get("ukf_projected", {}).get(camera.name)
        if not projected:
            continue
        if show_pixel:
            points = [image_plane_point(camera, p[0], p[1], FRUSTUM_DEPTH_M) for p in projected["projected_points"] if p is not None]
            if points:
                arr = np.asarray(points)
                fig.add_trace(go.Scatter3d(x=arr[:, 0], y=arr[:, 1], z=arr[:, 2], mode="markers", marker={"size": 4, "color": UKF_COLOR, "symbol": "circle-open"}, name=f"{camera.name} UKF projected sigma points", showlegend=False))
        mean_uv = projected.get("measurement_mean")
        cov_uv = projected.get("measurement_covariance")
        if show_covariance_pixel and mean_uv is not None and cov_uv is not None:
            add_image_marker(fig, camera, mean_uv, color=UKF_MEASUREMENT_COLOR, name=f"{camera.name} UKF sigma mean", size=5)
            add_pixel_covariance_ellipses(fig, camera, mean_uv, cov_uv, PROJECTION_SIGMA_LEVELS, color=UKF_MEASUREMENT_COLOR, name_prefix="UKF sigma covariance")


def add_ekf_jacobian_covariance(fig: go.Figure, filter_record: dict[str, Any]) -> None:
    for camera in selected_cameras:
        projected = filter_record.get("ekf_projected", {}).get(camera.name)
        if not projected:
            continue
        mean_uv = projected.get("measurement_mean")
        cov_uv = projected.get("measurement_covariance")
        if mean_uv is not None and cov_uv is not None:
            add_image_marker(fig, camera, mean_uv, color=EKF_COLOR, name=f"{camera.name} EKF Jacobian mean", size=5)
            add_pixel_covariance_ellipses(fig, camera, mean_uv, cov_uv, PROJECTION_SIGMA_LEVELS, color=EKF_COLOR, name_prefix="EKF Jacobian covariance")


def add_triangulation_overlays(fig: go.Figure, record: dict[str, Any], *, show_rays: bool, show_point: bool, show_covariance: bool, show_samples: bool, show_sample_rays: bool) -> None:
    if show_rays:
        for observation in observations_for_detections(record["detections"]):
            camera = cameras_by_id[observation.camera_id]
            origin, direction = camera.pixel_to_world_ray(observation.u, observation.v)
            endpoint = origin + TRIANGULATION_DETECTION_RAY_LENGTH_M * direction
            fig.add_trace(line_trace([origin, endpoint], color=KF_COLOR, width=4, name=f"{camera.name} triangulation detection ray"))
    result = record.get("triangulation_result")
    if show_point and result is not None and result.point is not None:
        point = result.point
        fig.add_trace(go.Scatter3d(x=[point[0]], y=[point[1]], z=[point[2]], mode="markers", marker={"size": 8, "color": TRIANGULATION_POINT_COLOR, "symbol": "circle"}, name="triangulation point", hovertemplate="triangulation point<br>x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}<extra></extra>"))
    if show_covariance and result is not None and result.point is not None and result.covariance is not None:
        x, y, z, _ = covariance_ellipsoid_mesh(result.point, result.covariance, POSITION_ELLIPSOID_SIGMA, num_u=28, num_v=14)
        fig.add_trace(go.Surface(x=x, y=y, z=z, surfacecolor=np.ones_like(x), colorscale=[[0.0, KF_COVARIANCE_COLOR], [1.0, KF_COVARIANCE_COLOR]], opacity=0.24, showscale=False, name="triangulation Monte Carlo covariance", showlegend=False))
    samples = record.get("triangulation_samples", np.empty((0, 3)))
    if show_samples and samples.size:
        fig.add_trace(go.Scatter3d(x=samples[:, 0], y=samples[:, 1], z=samples[:, 2], mode="markers", marker={"size": 4, "color": KF_COVARIANCE_COLOR, "opacity": 0.65}, name="Monte Carlo triangulation samples"))
    if show_sample_rays and samples.size:
        for camera in selected_cameras:
            origin = camera.extrinsics.t_camera_to_world
            points: list[np.ndarray | None] = []
            for sample in samples:
                points.extend([origin, sample, None])
            fig.add_trace(line_trace(points, color=KF_COVARIANCE_COLOR, width=1.0, name=f"{camera.name} rays to MC samples", opacity=0.45))


In [ ]:
def build_unified_static_figure(
    record: dict[str, Any],
    *,
    title: str = "Unified Filter Visualization",
    show_camera_geometry: bool = False,
    show_ground_truth_position: bool = False,
    show_ground_truth_velocity: bool = False,
    show_ground_truth_rays: bool = False,
    show_ground_truth_projection_points: bool = False,
    show_ground_truth_projection_covariance: bool = False,
    show_kf_prior_position: bool = False,
    show_kf_prior_velocity: bool = False,
    show_kf_prior_covariance: bool = False,
    show_kf_prior_rays: bool = False,
    show_kf_prior_projection_points: bool = False,
    show_kf_prior_projection_covariance: bool = False,
    show_ekf_prior_position: bool = False,
    show_ekf_prior_velocity: bool = False,
    show_ekf_prior_covariance: bool = False,
    show_ekf_prior_rays: bool = False,
    show_ekf_prior_projection_points: bool = False,
    show_ekf_prior_projection_covariance: bool = False,
    show_ukf_prior_position: bool = False,
    show_ukf_prior_velocity: bool = False,
    show_ukf_prior_covariance: bool = False,
    show_ukf_prior_rays: bool = False,
    show_ukf_prior_projection_points: bool = False,
    show_ukf_prior_projection_covariance: bool = False,
    show_ukf_sigma_points_3d: bool = False,
    show_ukf_sigma_points_pixel: bool = False,
    show_ukf_sigma_covariance_pixel: bool = False,
    show_ekf_jacobian_covariance_pixel: bool = False,
    show_kf_posterior_position: bool = False,
    show_kf_posterior_velocity: bool = False,
    show_kf_posterior_covariance: bool = False,
    show_kf_posterior_rays: bool = False,
    show_kf_posterior_projection_points: bool = False,
    show_ekf_posterior_position: bool = False,
    show_ekf_posterior_velocity: bool = False,
    show_ekf_posterior_covariance: bool = False,
    show_ekf_posterior_rays: bool = False,
    show_ekf_posterior_projection_points: bool = False,
    show_ukf_posterior_position: bool = False,
    show_ukf_posterior_velocity: bool = False,
    show_ukf_posterior_covariance: bool = False,
    show_ukf_posterior_rays: bool = False,
    show_ukf_posterior_projection_points: bool = False,
    show_kf_previous_posterior_position: bool = False,
    show_kf_previous_posterior_velocity: bool = False,
    show_kf_previous_posterior_covariance: bool = False,
    show_kf_previous_to_prior_line: bool = False,
    show_ekf_previous_posterior_position: bool = False,
    show_ekf_previous_posterior_velocity: bool = False,
    show_ekf_previous_posterior_covariance: bool = False,
    show_ekf_previous_to_prior_line: bool = False,
    show_ukf_previous_posterior_position: bool = False,
    show_ukf_previous_posterior_velocity: bool = False,
    show_ukf_previous_posterior_covariance: bool = False,
    show_ukf_previous_to_prior_line: bool = False,
    show_measurement_points: bool = False,
    show_measurement_pixel_cells: bool = False,
    show_measurement_covariance: bool = False,
    show_triangulation_rays: bool = False,
    show_triangulation_point: bool = False,
    show_triangulation_covariance: bool = False,
    show_triangulation_samples: bool = False,
    show_triangulation_sample_rays: bool = False,
) -> go.Figure:
    fig = go.Figure()
    if show_camera_geometry:
        add_camera_geometry(fig, selected_cameras)

    add_ground_truth_overlays(
        fig,
        record,
        show_position=show_ground_truth_position,
        show_velocity=show_ground_truth_velocity,
        show_rays=show_ground_truth_rays,
        show_projection_points=show_ground_truth_projection_points,
        show_projection_covariance=show_ground_truth_projection_covariance,
    )
    add_detection_measurements(
        fig,
        record,
        show_points=show_measurement_points,
        show_pixel_cells=show_measurement_pixel_cells,
        show_covariance=show_measurement_covariance,
    )
    add_triangulation_overlays(
        fig,
        record,
        show_rays=show_triangulation_rays,
        show_point=show_triangulation_point,
        show_covariance=show_triangulation_covariance,
        show_samples=show_triangulation_samples,
        show_sample_rays=show_triangulation_sample_rays,
    )
    kf_record = record["filters"].get("kf", empty_filter_record())
    if kf_record.get("valid"):
        add_filter_state_3d(fig, "kf", kf_record, "previous_posterior_state", "previous_posterior_covariance", position=show_kf_previous_posterior_position, velocity=show_kf_previous_posterior_velocity, covariance=show_kf_previous_posterior_covariance, label="previous posterior", color=PREVIOUS_ALPHA_COLOR)
        if show_kf_previous_to_prior_line:
            add_previous_to_prior_line(fig, "kf", kf_record, PREVIOUS_ALPHA_COLOR)
        add_filter_state_3d(fig, "kf", kf_record, "prior_state", "prior_covariance", position=show_kf_prior_position, velocity=show_kf_prior_velocity, covariance=show_kf_prior_covariance, label="prior", color=KF_COLOR)
        add_filter_projection_overlays(fig, "kf", kf_record, "prior_state", "prior_covariance", show_rays=show_kf_prior_rays, show_points=show_kf_prior_projection_points, show_covariance=show_kf_prior_projection_covariance, color=KF_COLOR, label="prior")
        add_filter_state_3d(fig, "kf", kf_record, "posterior_state", "posterior_covariance", position=show_kf_posterior_position, velocity=show_kf_posterior_velocity, covariance=show_kf_posterior_covariance, label="posterior", color=KF_COLOR)
        add_filter_projection_overlays(fig, "kf", kf_record, "posterior_state", "posterior_covariance", show_rays=show_kf_posterior_rays, show_points=show_kf_posterior_projection_points, show_covariance=False, color=KF_COLOR, label="posterior")
    ekf_record = record["filters"].get("ekf", empty_filter_record())
    if ekf_record.get("valid"):
        add_filter_state_3d(fig, "ekf", ekf_record, "previous_posterior_state", "previous_posterior_covariance", position=show_ekf_previous_posterior_position, velocity=show_ekf_previous_posterior_velocity, covariance=show_ekf_previous_posterior_covariance, label="previous posterior", color=PREVIOUS_ALPHA_COLOR)
        if show_ekf_previous_to_prior_line:
            add_previous_to_prior_line(fig, "ekf", ekf_record, PREVIOUS_ALPHA_COLOR)
        add_filter_state_3d(fig, "ekf", ekf_record, "prior_state", "prior_covariance", position=show_ekf_prior_position, velocity=show_ekf_prior_velocity, covariance=show_ekf_prior_covariance, label="prior", color=EKF_COLOR)
        add_filter_projection_overlays(fig, "ekf", ekf_record, "prior_state", "prior_covariance", show_rays=show_ekf_prior_rays, show_points=show_ekf_prior_projection_points, show_covariance=show_ekf_prior_projection_covariance, color=EKF_COLOR, label="prior")
        add_filter_state_3d(fig, "ekf", ekf_record, "posterior_state", "posterior_covariance", position=show_ekf_posterior_position, velocity=show_ekf_posterior_velocity, covariance=show_ekf_posterior_covariance, label="posterior", color=EKF_COLOR)
        add_filter_projection_overlays(fig, "ekf", ekf_record, "posterior_state", "posterior_covariance", show_rays=show_ekf_posterior_rays, show_points=show_ekf_posterior_projection_points, show_covariance=False, color=EKF_COLOR, label="posterior")
    ukf_record = record["filters"].get("ukf", empty_filter_record())
    if ukf_record.get("valid"):
        add_filter_state_3d(fig, "ukf", ukf_record, "previous_posterior_state", "previous_posterior_covariance", position=show_ukf_previous_posterior_position, velocity=show_ukf_previous_posterior_velocity, covariance=show_ukf_previous_posterior_covariance, label="previous posterior", color=PREVIOUS_ALPHA_COLOR)
        if show_ukf_previous_to_prior_line:
            add_previous_to_prior_line(fig, "ukf", ukf_record, PREVIOUS_ALPHA_COLOR)
        add_filter_state_3d(fig, "ukf", ukf_record, "prior_state", "prior_covariance", position=show_ukf_prior_position, velocity=show_ukf_prior_velocity, covariance=show_ukf_prior_covariance, label="prior", color=UKF_COLOR)
        add_filter_projection_overlays(fig, "ukf", ukf_record, "prior_state", "prior_covariance", show_rays=show_ukf_prior_rays, show_points=show_ukf_prior_projection_points, show_covariance=show_ukf_prior_projection_covariance, color=UKF_COLOR, label="prior")
        add_filter_state_3d(fig, "ukf", ukf_record, "posterior_state", "posterior_covariance", position=show_ukf_posterior_position, velocity=show_ukf_posterior_velocity, covariance=show_ukf_posterior_covariance, label="posterior", color=UKF_COLOR)
        add_filter_projection_overlays(fig, "ukf", ukf_record, "posterior_state", "posterior_covariance", show_rays=show_ukf_posterior_rays, show_points=show_ukf_posterior_projection_points, show_covariance=False, color=UKF_COLOR, label="posterior")

    ukf_record = record["filters"].get("ukf", empty_filter_record())
    if ukf_record.get("valid"):
        add_ukf_sigma_overlays(
            fig,
            ukf_record,
            show_3d=show_ukf_sigma_points_3d,
            show_pixel=show_ukf_sigma_points_pixel,
            show_covariance_pixel=show_ukf_sigma_covariance_pixel,
        )
    ekf_record = record["filters"].get("ekf", empty_filter_record())
    if ekf_record.get("valid") and show_ekf_jacobian_covariance_pixel:
        add_ekf_jacobian_covariance(fig, ekf_record)

    apply_layout(fig, title, record["timestamp"])
    return fig


## Example Figure Command

Everything configurable is written explicitly below and disabled by default. Copy/paste this cell and turn on only the layers you need.


In [ ]:
fig_static_1 = build_unified_static_figure(
    record_for_timestep(),
    title="1. Custom Unified Static Figure",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_prior_position=False,
    show_kf_prior_velocity=False,
    show_kf_prior_covariance=False,
    show_kf_prior_rays=False,
    show_kf_prior_projection_points=False,
    show_kf_prior_projection_covariance=False,
    show_ekf_prior_position=False,
    show_ekf_prior_velocity=False,
    show_ekf_prior_covariance=False,
    show_ekf_prior_rays=False,
    show_ekf_prior_projection_points=False,
    show_ekf_prior_projection_covariance=False,
    show_ukf_prior_position=False,
    show_ukf_prior_velocity=False,
    show_ukf_prior_covariance=False,
    show_ukf_prior_rays=False,
    show_ukf_prior_projection_points=False,
    show_ukf_prior_projection_covariance=False,
    show_ukf_sigma_points_3d=False,
    show_ukf_sigma_points_pixel=False,
    show_ukf_sigma_covariance_pixel=False,
    show_ekf_jacobian_covariance_pixel=False,
    show_kf_posterior_position=False,
    show_kf_posterior_velocity=False,
    show_kf_posterior_covariance=False,
    show_kf_posterior_rays=False,
    show_kf_posterior_projection_points=False,
    show_ekf_posterior_position=False,
    show_ekf_posterior_velocity=False,
    show_ekf_posterior_covariance=False,
    show_ekf_posterior_rays=False,
    show_ekf_posterior_projection_points=False,
    show_ukf_posterior_position=False,
    show_ukf_posterior_velocity=False,
    show_ukf_posterior_covariance=False,
    show_ukf_posterior_rays=False,
    show_ukf_posterior_projection_points=False,
    show_kf_previous_posterior_position=False,
    show_kf_previous_posterior_velocity=False,
    show_kf_previous_posterior_covariance=False,
    show_kf_previous_to_prior_line=False,
    show_ekf_previous_posterior_position=False,
    show_ekf_previous_posterior_velocity=False,
    show_ekf_previous_posterior_covariance=False,
    show_ekf_previous_to_prior_line=False,
    show_ukf_previous_posterior_position=False,
    show_ukf_previous_posterior_velocity=False,
    show_ukf_previous_posterior_covariance=False,
    show_ukf_previous_to_prior_line=False,
    show_measurement_points=False,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=False,
    show_triangulation_rays=False,
    show_triangulation_point=False,
    show_triangulation_covariance=False,
    show_triangulation_samples=False,
    show_triangulation_sample_rays=False,
)
fig_static_1.show()


In [ ]:
fig_static_2 = build_unified_static_figure(
    record_for_timestep(),
    title="2. Ground truth position",
    show_camera_geometry=True,
    show_ground_truth_position=True,
    show_ground_truth_velocity=False,
)
fig_static_2.show()


In [ ]:
fig_static_3 = build_unified_static_figure(
    record_for_timestep(),
    title="3. Ground truth state with velocity",
    show_camera_geometry=True,
    show_ground_truth_position=True,
    show_ground_truth_velocity=True
)
fig_static_3.show()


In [ ]:
fig_static_4 = build_unified_static_figure(
    record_for_timestep(),
    title="4. Ideal ground truth projection",
    show_camera_geometry=True,
    show_ground_truth_position=True,
    show_ground_truth_velocity=True,
    show_ground_truth_rays=True,
    show_ground_truth_projection_points=True,
    show_ground_truth_projection_covariance=False,
)
fig_static_4.show()


In [ ]:
fig_static_5 = build_unified_static_figure(
    record_for_timestep(),
    title="5. Ideal ground truth projection with covariance",
    show_camera_geometry=True,
    show_ground_truth_position=True,
    show_ground_truth_velocity=True,
    show_ground_truth_rays=True,
    show_ground_truth_projection_points=True,
    show_ground_truth_projection_covariance=True,
)
fig_static_5.show()


In [ ]:
fig_static_6 = build_unified_static_figure(
    record_for_timestep(),
    title="6. Detection measurements",
    show_camera_geometry=True,
    show_ground_truth_position=True,
    show_ground_truth_velocity=True,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=True,
    show_measurement_points=True,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=False,
)
fig_static_6.show()


In [ ]:
fig_static_7 = build_unified_static_figure(
    record_for_timestep(),
    title="7. Detection measurements with pixel cells",
    show_camera_geometry=True,
    show_ground_truth_position=True,
    show_ground_truth_velocity=True,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=True,
    show_measurement_points=True,
    show_measurement_pixel_cells=True,
    show_measurement_covariance=False,
)
fig_static_7.show()

In [ ]:
fig_static_8 = build_unified_static_figure(
    record_for_timestep(),
    title="8. Detection measurements with pixel cells",
    show_camera_geometry=True,
    show_ground_truth_position=True,
    show_ground_truth_velocity=True,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=True,
    show_measurement_points=True,
    show_measurement_pixel_cells=True,
    show_measurement_covariance=True,
)
fig_static_8.show()

In [ ]:
fig_static_9 = build_unified_static_figure(
    record_for_timestep(),
    title="9. Detection measurements with pixel cells",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_measurement_points=True,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=True,
)
fig_static_9.show()

In [ ]:
fig_static_10 = build_unified_static_figure(
    record_for_timestep(),
    title="10. Triangulation rays through detections.",
    show_camera_geometry=True,
    show_measurement_points=True,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=True,
    show_triangulation_rays=True,
    show_triangulation_point=False,
    show_triangulation_samples=False,
    show_triangulation_sample_rays=False,
)
fig_static_10.show()


In [ ]:
fig_static_11 = build_unified_static_figure(
    record_for_timestep(),
    title="11. Triangulation rays and point",
    show_camera_geometry=True,
    show_measurement_points=True,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=True,
    show_triangulation_rays=True,
    show_triangulation_point=True,
    show_triangulation_samples=False,
    show_triangulation_sample_rays=False,
)
fig_static_11.show()


In [ ]:
fig_static_12 = build_unified_static_figure(
    record_for_timestep(),
    title="12. Triangulation monte carlo samples and rays",
    show_camera_geometry=True,
    show_measurement_points=True,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=True,
    show_triangulation_rays=False,
    show_triangulation_point=True,
    show_triangulation_samples=True,
    show_triangulation_sample_rays=True,
)
fig_static_12.show()


In [ ]:
fig_static_13 = build_unified_static_figure(
    record_for_timestep(),
    title="13. Triangulation monte carlo samples and rays with covariance",
    show_camera_geometry=True,
    show_measurement_points=True,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=True,
    show_triangulation_rays=False,
    show_triangulation_point=True,
    show_triangulation_samples=True,
    show_triangulation_covariance=True,
    show_triangulation_sample_rays=True,
)
fig_static_13.show()

In [ ]:
fig_static_14 = build_unified_static_figure(
    record_for_timestep(),
    title="14. Triangulation point with covariance",
    show_camera_geometry=True,
    show_measurement_points=True,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=True,
    show_triangulation_rays=False,
    show_triangulation_point=True,
    show_triangulation_covariance=True,
    show_triangulation_samples=False,
    show_triangulation_sample_rays=False,
)
fig_static_14.show()

In [ ]:
fig_static_15 = build_unified_static_figure(
    record_for_timestep(),
    title="15. Ground truth, measurements, and triangulation point with covariance",
    show_camera_geometry=True,
    show_ground_truth_position=True,
    show_ground_truth_velocity=False,
    show_measurement_points=True,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=True,
    show_triangulation_rays=False,
    show_triangulation_point=True,
    show_triangulation_covariance=True,
    show_triangulation_samples=False,
    show_triangulation_sample_rays=False,
)
fig_static_15.show()

In [ ]:
fig_static_16 = build_unified_static_figure(
    record_for_timestep(),
    title="16. Previous posterior state",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ekf_prior_position=False,
    show_ekf_prior_velocity=False,
    show_ekf_prior_covariance=False,
    show_ekf_prior_rays=False,
    show_ekf_prior_projection_points=False,
    show_ekf_prior_projection_covariance=False,
    show_ekf_previous_posterior_position=True,
    show_ekf_previous_posterior_velocity=True,
    show_ekf_previous_posterior_covariance=True,
    show_ekf_previous_to_prior_line=False,
)
fig_static_16.show()

In [ ]:
fig_static_17 = build_unified_static_figure(
    record_for_timestep(),
    title="17. Previous posterior state and next prior state position",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ekf_prior_position=True,
    show_ekf_prior_velocity=False,
    show_ekf_prior_covariance=False,
    show_ekf_prior_rays=False,
    show_ekf_prior_projection_points=False,
    show_ekf_prior_projection_covariance=False,
    show_ekf_previous_posterior_position=True,
    show_ekf_previous_posterior_velocity=True,
    show_ekf_previous_posterior_covariance=True
)
fig_static_17.show()

In [ ]:
fig_static_18 = build_unified_static_figure(
    record_for_timestep(),
    title="18. Previous posterior state and next prior state position",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ekf_prior_position=True,
    show_ekf_prior_velocity=True,
    show_ekf_prior_covariance=True,
    show_ekf_prior_rays=False,
    show_ekf_prior_projection_points=False,
    show_ekf_prior_projection_covariance=False,
    show_ekf_previous_posterior_position=True,
    show_ekf_previous_posterior_velocity=False,
    show_ekf_previous_posterior_covariance=True,
    show_ekf_previous_to_prior_line=True
)
fig_static_18.show()

In [ ]:
fig_static_19 = build_unified_static_figure(
    record_for_timestep(),
    title="19. KF Prior with covariance",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_prior_position=True,
    show_kf_prior_velocity=True,
    show_kf_prior_covariance=True,
    show_kf_prior_rays=False,
    show_kf_prior_projection_points=False,
    show_kf_prior_projection_covariance=False,
)
fig_static_19.show()

In [ ]:
fig_static_20 = build_unified_static_figure(
    record_for_timestep(),
    title="20. KF after applied measurement model",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_prior_position=True,
    show_kf_prior_velocity=False,
    show_kf_prior_covariance=True,
    show_kf_prior_rays=False,
    show_kf_prior_projection_points=False,
    show_kf_prior_projection_covariance=False,
)
fig_static_20.show()

In [ ]:
fig_static_21 = build_unified_static_figure(
    record_for_timestep(),
    title="21. EKF Prior with covariance",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ekf_prior_position=True,
    show_ekf_prior_velocity=True,
    show_ekf_prior_covariance=True,
    show_ekf_prior_rays=False,
    show_ekf_prior_projection_points=False,
    show_ekf_prior_projection_covariance=False,
)
fig_static_21.show()

In [ ]:
fig_static_22 = build_unified_static_figure(
    record_for_timestep(),
    title="22. EKF Prior with covariance and projection",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ekf_prior_position=True,
    show_ekf_prior_velocity=True,
    show_ekf_prior_covariance=True,
    show_ekf_prior_rays=True,
    show_ekf_prior_projection_points=True,
    show_ekf_prior_projection_covariance=False,
)
fig_static_22.show()

In [ ]:
fig_static_23 = build_unified_static_figure(
    record_for_timestep(),
    title="23. EKF Prior with covariance and projection and projection covariance",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ekf_prior_position=True,
    show_ekf_prior_velocity=True,
    show_ekf_prior_covariance=True,
    show_ekf_prior_rays=False,
    show_ekf_prior_projection_points=True,
    show_ekf_prior_projection_covariance=True,
)
fig_static_23.show()

In [ ]:
fig_static_24 = build_unified_static_figure(
    record_for_timestep(),
    title="24. EKF after applied measurement model and jacobian covariance",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ekf_prior_position=False,
    show_ekf_prior_velocity=False,
    show_ekf_prior_covariance=False,
    show_ekf_prior_rays=False,
    show_ekf_prior_projection_points=True,
    show_ekf_prior_projection_covariance=True,
)
fig_static_24.show()

In [ ]:
fig_static_25 = build_unified_static_figure(
    record_for_timestep(),
    title="25. UKF Prior with covariance",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ukf_prior_position=True,
    show_ukf_prior_velocity=True,
    show_ukf_prior_covariance=True,
    show_ukf_prior_rays=False,
    show_ukf_prior_projection_points=False,
    show_ukf_prior_projection_covariance=False,
)
fig_static_25.show()

In [ ]:
fig_static_26 = build_unified_static_figure(
    record_for_timestep(),
    title="26. UKF Prior with covariance and sigma points",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ukf_prior_position=True,
    show_ukf_prior_velocity=True,
    show_ukf_prior_covariance=True,
    show_ukf_prior_rays=False,
    show_ukf_prior_projection_points=False,
    show_ukf_prior_projection_covariance=False,
    show_ukf_sigma_points_3d=True,
)
fig_static_26.show()

In [ ]:
fig_static_27 = build_unified_static_figure(
    record_for_timestep(),
    title="27. UKF Prior with covariance and sigma points and projected sigma points",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ukf_prior_position=True,
    show_ukf_prior_velocity=True,
    show_ukf_prior_covariance=True,
    show_ukf_prior_rays=True,
    show_ukf_prior_projection_points=True,
    show_ukf_prior_projection_covariance=False,
    show_ukf_sigma_points_3d=True,
    show_ukf_sigma_points_pixel=True,
    show_ukf_sigma_covariance_pixel=True
)
fig_static_27.show()

In [ ]:
fig_static_28 = build_unified_static_figure(
    record_for_timestep(),
    title="28. UKF Prior with covariance and projected mean with covariance",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ukf_prior_position=True,
    show_ukf_prior_velocity=True,
    show_ukf_prior_covariance=True,
    show_ukf_prior_rays=False,
    show_ukf_prior_projection_points=True,
    show_ukf_sigma_points_3d=False,
    show_ukf_sigma_points_pixel=False,
    show_ukf_sigma_covariance_pixel=True,
)
fig_static_28.show()

In [ ]:
fig_static_29 = build_unified_static_figure(
    record_for_timestep(),
    title="29. UKF after applied measurement model and covariance computation",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_ukf_prior_rays=False,
    show_ukf_prior_projection_points=True,
    show_ukf_sigma_points_3d=False,
    show_ukf_sigma_points_pixel=False,
    show_ukf_sigma_covariance_pixel=True,
)
fig_static_29.show()

In [ ]:
fig_static_30 = build_unified_static_figure(
    record_for_timestep(),
    title="30. KF Triangulation and Prior and measurement with covariances",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_prior_position=True,
    show_kf_prior_velocity=True,
    show_kf_prior_covariance=True,
    show_triangulation_point=True,
    show_triangulation_covariance=True,
    show_measurement_points=True,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=True
)
fig_static_30.show()

In [ ]:
fig_static_31 = build_unified_static_figure(
    record_for_timestep(),
    title="31. KF Posterior, triangulation point, measurement and prior",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_prior_position=True,
    show_kf_prior_velocity=True,
    show_kf_prior_covariance=True,
    show_triangulation_point=True,
    show_triangulation_covariance=True,
    show_kf_posterior_position=True,
    show_kf_posterior_velocity=True,
    show_kf_posterior_covariance=True,
    show_measurement_points=True,
    show_measurement_pixel_cells=False,
    show_measurement_covariance=True
)
fig_static_31.show()

In [ ]:
fig_static_32 = build_unified_static_figure(
    record_for_timestep(),
    title="32. KF Posterior, EKF Prior, EKF projection, and measurement with covariances",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_posterior_position=True,
    show_kf_posterior_velocity=True,
    show_kf_posterior_covariance=True,
    show_ekf_prior_position=True,
    show_ekf_prior_velocity=True,
    show_ekf_prior_covariance=True,
    show_ekf_prior_projection_points=True,
    show_ekf_prior_projection_covariance=True,
    show_measurement_points=True,
    show_measurement_covariance=True,
)
fig_static_32.show()

In [ ]:
fig_static_33 = build_unified_static_figure(
    record_for_timestep(),
    title="33. KF Posterior, EKF Prior, EKF projection, EKF posterior, and measurement with covariances",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_posterior_position=True,
    show_kf_posterior_velocity=True,
    show_kf_posterior_covariance=True,
    show_ekf_prior_position=True,
    show_ekf_prior_velocity=True,
    show_ekf_prior_covariance=True,
    show_ekf_prior_projection_points=True,
    show_ekf_prior_projection_covariance=True,
    show_ekf_posterior_position=True,
    show_ekf_posterior_velocity=True,
    show_ekf_posterior_covariance=True,
    show_measurement_points=True,
    show_measurement_covariance=True,
)
fig_static_33.show()

In [ ]:
fig_static_34 = build_unified_static_figure(
    record_for_timestep(),
    title="34. KF Posterior, EKF posterior, UKF prior, UKF projection, and measurement with covariances",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_posterior_position=True,
    show_kf_posterior_velocity=True,
    show_kf_posterior_covariance=True,
    show_ekf_posterior_position=True,
    show_ekf_posterior_velocity=True,
    show_ekf_posterior_covariance=True,
    show_ukf_prior_position=True,
    show_ukf_prior_velocity=True,
    show_ukf_prior_covariance=True,
    show_ukf_prior_projection_points=True,
    show_ukf_sigma_covariance_pixel=True,
    show_measurement_points=True,
    show_measurement_covariance=True,
)
fig_static_34.show()

In [ ]:
fig_static_35 = build_unified_static_figure(
    record_for_timestep(),
    title="35. KF Posterior, EKF posterior, UKF prior, UKF projection, UKF Posterior and measurement with covariances",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_posterior_position=True,
    show_kf_posterior_velocity=True,
    show_kf_posterior_covariance=True,
    show_ekf_posterior_position=True,
    show_ekf_posterior_velocity=True,
    show_ekf_posterior_covariance=True,
    show_ukf_prior_position=True,
    show_ukf_prior_velocity=True,
    show_ukf_prior_covariance=True,
    show_ukf_prior_projection_points=True,
    show_ukf_sigma_covariance_pixel=True,
    show_ukf_posterior_position=True,
    show_ukf_posterior_velocity=True,
    show_ukf_posterior_covariance=True,
    show_measurement_points=True,
    show_measurement_covariance=True,
)
fig_static_35.show()

In [ ]:
fig_static_36 = build_unified_static_figure(
    record_for_timestep(),
    title="36. KF Posterior, EKF posterior, UKF Posterior and measurement with covariances",
    show_camera_geometry=True,
    show_ground_truth_position=False,
    show_ground_truth_velocity=False,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_posterior_position=True,
    show_kf_posterior_velocity=True,
    show_kf_posterior_covariance=True,
    show_ekf_posterior_position=True,
    show_ekf_posterior_velocity=True,
    show_ekf_posterior_covariance=True,
    show_ukf_posterior_position=True,
    show_ukf_posterior_velocity=True,
    show_ukf_posterior_covariance=True,
    show_measurement_points=True,
    show_measurement_covariance=True,
)
fig_static_36.show()

In [ ]:
fig_static_37 = build_unified_static_figure(
    record_for_timestep(),
    title="37. KF Posterior, EKF posterior, UKF Posterior and Ground Truth with covariances",
    show_camera_geometry=True,
    show_ground_truth_position=True,
    show_ground_truth_velocity=True,
    show_ground_truth_rays=False,
    show_ground_truth_projection_points=False,
    show_ground_truth_projection_covariance=False,
    show_kf_posterior_position=True,
    show_kf_posterior_velocity=True,
    show_kf_posterior_covariance=True,
    show_ekf_posterior_position=True,
    show_ekf_posterior_velocity=True,
    show_ekf_posterior_covariance=True,
    show_ukf_posterior_position=True,
    show_ukf_posterior_velocity=True,
    show_ukf_posterior_covariance=True,
)
fig_static_37.show()

## Export Static Figures as HTML


In [ ]:
def safe_filename_part(value: str) -> str:
    cleaned = []
    for char in str(value).lower():
        if char.isalnum():
            cleaned.append(char)
        elif char in {" ", "-", "_", "."}:
            cleaned.append("_")
    name = "".join(cleaned).strip("_")
    while "__" in name:
        name = name.replace("__", "_")
    return name or "figure"


def static_figure_index(name: str) -> int:
    try:
        return int(name.rsplit("_", 1)[-1])
    except ValueError:
        return 9999


def figure_title_text(fig: go.Figure, fallback: str) -> str:
    title = fig.layout.title.text
    return str(title) if title else fallback


def export_static_figures_to_html(output_dir: Path = STATIC_HTML_OUTPUT_DIR) -> pd.DataFrame:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    static_figures = [(name, value) for name, value in globals().items() if name.startswith("fig_static_") and isinstance(value, go.Figure)]
    static_figures.sort(key=lambda item: static_figure_index(item[0]))
    rows = []
    for name, fig in static_figures:
        index = static_figure_index(name)
        title = figure_title_text(fig, name)
        filename = f"{index:02d}_{safe_filename_part(title)}.html"
        path = output_dir / filename
        fig.write_html(path, include_plotlyjs=STATIC_HTML_INCLUDE_PLOTLYJS, full_html=True)
        try:
            display_path = path.relative_to(REPO_ROOT)
        except ValueError:
            display_path = path
        rows.append({"figure": name, "title": title, "path": str(display_path)})
    return pd.DataFrame(rows)


static_html_exports = export_static_figures_to_html()
display(static_html_exports)
